# 13 GHSL Six-City 2015 and 2020 Model Evaluation

This notebook evaluates the already trained Notebook 12 models against actual GHSL observations for the existing six cities.

It compares:

- **2010→2015 validation:** model prediction for 2015 versus actual 2015 built surface
- **2015→2020 test:** model prediction for 2020 versus actual 2020 built surface
- Linear Regression
- Random Forest
- Persistence baseline, where predicted next built surface equals current built surface

The notebook does **not** retrain models and does **not** use Jaipur. All Spark and HDFS operations run on EC2 through `/content/drive/MyDrive/remote_exec.sh`.

Run one cell at a time.

In [1]:
# ============================================================
# CELL 1: DRIVE, PATHS, AND CONSTANTS
# ============================================================
from google.colab import drive
from pathlib import Path
import base64, json, os, shutil, subprocess, time

drive.mount('/content/drive', force_remount=False)
REMOTE_EXEC=Path('/content/drive/MyDrive/remote_exec.sh')
DRIVE_SSH_KEY=Path('/content/drive/MyDrive/hadoop-hdfs.pem')
COLAB_SSH_KEY=Path('/content/hadoop-hdfs.pem')
MANIFEST12=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/model_training_v1/training_manifest.json')
ARTIFACT_ROOT=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_evaluation_v1')
for p in [REMOTE_EXEC,DRIVE_SSH_KEY,MANIFEST12]:
    if not p.is_file(): raise FileNotFoundError(str(p))
REMOTE_EXEC.chmod(0o700)
if COLAB_SSH_KEY.exists(): COLAB_SSH_KEY.unlink()
shutil.copy2(DRIVE_SSH_KEY,COLAB_SSH_KEY); COLAB_SSH_KEY.chmod(0o600)
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
MANIFEST=json.loads(MANIFEST12.read_text(encoding='utf-8'))
TRANSITIONS='/data/ghsl/modeling/six_city_transitions_v1'
MODEL_ROOT='/data/ghsl/modeling/models/six_city_growth_v1'
LR_MODEL=MODEL_ROOT+'/linear_regression'
RF_MODEL=MODEL_ROOT+'/random_forest'
EVAL_ROOT='/data/ghsl/modeling/evaluation/six_city_2015_2020_v1'
LR_PRED=EVAL_ROOT+'/linear_regression_predictions'
RF_PRED=EVAL_ROOT+'/random_forest_predictions'
SUMMARY_OUT=EVAL_ROOT+'/metric_tables'
EXPECTED_CITIES=['bengaluru','chennai_metro','delhi_ncr','hyderabad','mumbai_metro','pune_metro']
print('Artifact root:',ARTIFACT_ROOT)
print('Jaipur included: NO')
print('CELL 1: SUCCESS')

Mounted at /content/drive
Artifact root: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_evaluation_v1
Jaipur included: NO
CELL 1: SUCCESS


In [2]:
# ============================================================
# CELL 2: REMOTE_EXEC.SH HELPER AND CONNECTION TEST
# ============================================================
def run_remote(command,timeout=300,show_output=True,raise_on_error=True):
    r=subprocess.run([str(REMOTE_EXEC),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:'); print(r.stderr)
    if raise_on_error and r.returncode!=0: raise RuntimeError('Remote command failed with return code '+str(r.returncode))
    return r
r=run_remote("set -e; echo REMOTE_EXEC_CONNECTION_SUCCESS; whoami; hostname",90)
if 'REMOTE_EXEC_CONNECTION_SUCCESS' not in r.stdout: raise RuntimeError('Connection marker missing')
print('CELL 2: SUCCESS')

REMOTE_EXEC_CONNECTION_SUCCESS
ubuntu
ip-172-31-30-94

REMOTE STDERR:

CELL 2: SUCCESS


In [3]:
# ============================================================
# CELL 3: HDFS, SPARK, AND MODEL PREFLIGHT
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local
for d in NameNode DataNode SecondaryNameNode; do jps | awk '{print $2}' | grep -qx "$d"; done
hdfs dfsadmin -safemode get | grep -qi 'Safe mode is OFF'
for p in /data/ghsl/modeling/six_city_transitions_v1 /data/ghsl/modeling/models/six_city_growth_v1/linear_regression /data/ghsl/modeling/models/six_city_growth_v1/random_forest; do hdfs dfs -test -e "$p"; done
python3 - <<'PY_REMOTE'
import setuptools,distutils
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
print('PYSPARK_ML_IMPORTS_PASS')
PY_REMOTE
echo SIX_CITY_EVALUATION_PREFLIGHT_SUCCESS
"""
r=run_remote(cmd,300)
if 'SIX_CITY_EVALUATION_PREFLIGHT_SUCCESS' not in r.stdout: raise RuntimeError('Preflight failed')
print('CELL 3: SUCCESS')

PYSPARK_ML_IMPORTS_PASS
SIX_CITY_EVALUATION_PREFLIGHT_SUCCESS

REMOTE STDERR:

CELL 3: SUCCESS


In [4]:
# ============================================================
# CELL 4: VALIDATE NOTEBOOK 12 PROVENANCE
# ============================================================
if MANIFEST.get('jaipur_used_in_training') is not False: raise RuntimeError('Jaipur contamination detected')
if MANIFEST.get('target_source_column')!='built_surface_total_m2': raise RuntimeError('Unexpected target')
results=MANIFEST['results']
if results['split_counts']!={'train':205800,'validation':29400,'test':29400}: raise RuntimeError('Unexpected split counts')
if len(results['features'])!=31: raise RuntimeError('Expected 31 features')
print('Training through: 2010')
print('Validation target: 2015')
print('Test target: 2020')
print('Existing six-city model metrics loaded: YES')
print('Jaipur included: NO')
print('CELL 4: SUCCESS')

Training through: 2010
Validation target: 2015
Test target: 2020
Existing six-city model metrics loaded: YES
Jaipur included: NO
CELL 4: SUCCESS


In [5]:
# ============================================================
# CELL 5: AUDIT THE 2015 AND 2020 EVALUATION SUBSETS
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
S=/tmp/nb13_eval_audit.py
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
import json
s=SparkSession.builder.appName('GHSL-Six-City-Evaluation-Audit').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 d=s.read.parquet('/data/ghsl/modeling/six_city_transitions_v1').filter(F.col('to_epoch').isin(2015,2020))
 rows=d.count(); keys=d.select('city_slug','grid_cell_id','from_epoch','to_epoch').distinct().count()
 by_epoch={int(r['to_epoch']):r['count'] for r in d.groupBy('to_epoch').count().collect()}
 by_city_epoch=[r.asDict() for r in d.groupBy('city_slug','to_epoch').count().orderBy('to_epoch','city_slug').collect()]
 null_labels=d.filter(F.col('label').isNull()|F.isnan('label')).count()
 if rows!=58800 or keys!=58800 or by_epoch!={2015:29400,2020:29400} or null_labels!=0: raise RuntimeError(repr({'rows':rows,'keys':keys,'by_epoch':by_epoch,'null_labels':null_labels}))
 if any(x['count']!=4900 for x in by_city_epoch): raise RuntimeError('Expected 4900 rows per city-period')
 out={'rows':rows,'keys':keys,'by_epoch':by_epoch,'by_city_epoch':by_city_epoch,'null_labels':null_labels}
 print('EVAL_AUDIT_JSON_BEGIN');print(json.dumps(out,sort_keys=True));print('EVAL_AUDIT_JSON_END');print('SIX_CITY_2015_2020_AUDIT_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 4g --conf spark.ui.enabled=false "$S"
rm -f "$S"
"""
r=run_remote(cmd,3600,show_output=False)
if 'SIX_CITY_2015_2020_AUDIT_SUCCESS' not in r.stdout:
    print(r.stdout[-12000:]);print(r.stderr[-6000:]);raise RuntimeError('Audit failed')
AUDIT=json.loads(r.stdout.split('EVAL_AUDIT_JSON_BEGIN',1)[1].split('EVAL_AUDIT_JSON_END',1)[0].strip())
print(json.dumps(AUDIT,indent=2))
print('CELL 5: SUCCESS')

{
  "by_city_epoch": [
    {
      "city_slug": "bengaluru",
      "count": 4900,
      "to_epoch": 2015
    },
    {
      "city_slug": "chennai_metro",
      "count": 4900,
      "to_epoch": 2015
    },
    {
      "city_slug": "delhi_ncr",
      "count": 4900,
      "to_epoch": 2015
    },
    {
      "city_slug": "hyderabad",
      "count": 4900,
      "to_epoch": 2015
    },
    {
      "city_slug": "mumbai_metro",
      "count": 4900,
      "to_epoch": 2015
    },
    {
      "city_slug": "pune_metro",
      "count": 4900,
      "to_epoch": 2015
    },
    {
      "city_slug": "bengaluru",
      "count": 4900,
      "to_epoch": 2020
    },
    {
      "city_slug": "chennai_metro",
      "count": 4900,
      "to_epoch": 2020
    },
    {
      "city_slug": "delhi_ncr",
      "count": 4900,
      "to_epoch": 2020
    },
    {
      "city_slug": "hyderabad",
      "count": 4900,
      "to_epoch": 2020
    },
    {
      "city_slug": "mumbai_metro",
      "count": 4900,
      "to_epo

In [6]:
# ============================================================
# CELL 6: SCORE LINEAR REGRESSION, RANDOM FOREST, AND PERSISTENCE
# Writes cell-level predictions without modifying source data.
# ============================================================
payload={'input':TRANSITIONS,'lr_model':LR_MODEL,'rf_model':RF_MODEL,'lr_out':LR_PRED,'rf_out':RF_PRED,'summary_out':SUMMARY_OUT}
payload_b64=base64.b64encode(json.dumps(payload).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
export SETUPTOOLS_USE_DISTUTILS=local
P=/tmp/nb13_score_payload.json; S=/tmp/nb13_score_models.py
echo __PAYLOAD__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
import json,sys
p=json.load(open(sys.argv[1]));s=SparkSession.builder.appName('GHSL-Six-City-2015-2020-Scoring').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration())
 for x in [p['lr_out'],p['rf_out'],p['summary_out']]:
  if fs.exists(s._jvm.org.apache.hadoop.fs.Path(x)): raise RuntimeError('Refusing overwrite: '+x)
 d=s.read.parquet(p['input']).filter(F.col('to_epoch').isin(2015,2020))
 keep=['city_slug','grid_cell_id','from_epoch','to_epoch','label','current_built_surface_m2','built_surface_growth_m2','grid_1km_row','grid_1km_col','grid_min_x','grid_max_x','grid_min_y','grid_max_y','geometry_wkb','geometry_crs']
 lr=PipelineModel.load(p['lr_model']).transform(d).withColumnRenamed('prediction','model_prediction')
 rf=PipelineModel.load(p['rf_model']).transform(d).withColumnRenamed('prediction','model_prediction')
 def finish(x,name):
  return (x.withColumn('model_name',F.lit(name)).withColumn('persistence_prediction',F.col('current_built_surface_m2')).withColumn('model_error',F.col('model_prediction')-F.col('label')).withColumn('model_abs_error',F.abs('model_error')).withColumn('persistence_error',F.col('persistence_prediction')-F.col('label')).withColumn('persistence_abs_error',F.abs('persistence_error')).withColumn('actual_growth_m2',F.col('label')-F.col('current_built_surface_m2')).withColumn('predicted_growth_m2',F.col('model_prediction')-F.col('current_built_surface_m2')).withColumn('growth_error_m2',F.col('predicted_growth_m2')-F.col('actual_growth_m2')).select(*(keep+['model_name','model_prediction','persistence_prediction','model_error','model_abs_error','persistence_error','persistence_abs_error','actual_growth_m2','predicted_growth_m2','growth_error_m2'])))
 lr=finish(lr,'linear_regression');rf=finish(rf,'random_forest')
 lr.repartition('to_epoch','city_slug').write.mode('errorifexists').partitionBy('to_epoch','city_slug').parquet(p['lr_out'])
 rf.repartition('to_epoch','city_slug').write.mode('errorifexists').partitionBy('to_epoch','city_slug').parquet(p['rf_out'])
 def metrics(df,pred_col):
  q=df.withColumn('prediction_eval',F.col(pred_col)); return {m:RegressionEvaluator(labelCol='label',predictionCol='prediction_eval',metricName=m).evaluate(q) for m in ['mae','rmse','r2']}
 summaries=[]
 for name,x in [('linear_regression',lr),('random_forest',rf)]:
  for epoch in [2015,2020]:
   e=x.filter(F.col('to_epoch')==epoch); summaries.append({'model':name,'scope':'overall','city_slug':'ALL','to_epoch':epoch,**metrics(e,'model_prediction')})
   if name=='linear_regression': summaries.append({'model':'persistence','scope':'overall','city_slug':'ALL','to_epoch':epoch,**metrics(e,'persistence_prediction')})
   for city in sorted(r[0] for r in e.select('city_slug').distinct().collect()): summaries.append({'model':name,'scope':'city','city_slug':city,'to_epoch':epoch,**metrics(e.filter(F.col('city_slug')==city),'model_prediction')})
   if name=='linear_regression':
    for city in sorted(r[0] for r in e.select('city_slug').distinct().collect()): summaries.append({'model':'persistence','scope':'city','city_slug':city,'to_epoch':epoch,**metrics(e.filter(F.col('city_slug')==city),'persistence_prediction')})
 summary=s.createDataFrame(summaries);summary.write.mode('errorifexists').parquet(p['summary_out'])
 print('METRICS_JSON_BEGIN');print(json.dumps(summaries,sort_keys=True));print('METRICS_JSON_END');print('SIX_CITY_SCORING_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=24 --conf spark.driver.maxResultSize=1g "$S" "$P"
rm -f "$S" "$P"
""".replace('__PAYLOAD__',payload_b64)
r=run_remote(cmd,7200,show_output=False)
if 'SIX_CITY_SCORING_SUCCESS' not in r.stdout:
    print(r.stdout[-16000:]);print(r.stderr[-8000:]);raise RuntimeError('Scoring failed')
METRICS=json.loads(r.stdout.split('METRICS_JSON_BEGIN',1)[1].split('METRICS_JSON_END',1)[0].strip())
print('Metric records:',len(METRICS))
print('CELL 6: SUCCESS')

Metric records: 42
CELL 6: SUCCESS


In [7]:
# ============================================================
# DISPLAY CELL 6 METRICS
# ============================================================

import pandas as pd

metrics_df = pd.DataFrame(
    METRICS
)

metrics_df = metrics_df[
    [
        "to_epoch",
        "scope",
        "city_slug",
        "model",
        "mae",
        "rmse",
        "r2",
    ]
].sort_values(
    [
        "to_epoch",
        "scope",
        "city_slug",
        "rmse",
    ]
).reset_index(
    drop=True
)

print(
    "Total metric records:",
    len(metrics_df),
)

print()
print(
    "========== OVERALL 2015 AND 2020 METRICS =========="
)

display(
    metrics_df[
        metrics_df[
            "scope"
        ]
        == "overall"
    ].reset_index(
        drop=True
    )
)

print()
print(
    "========== CITY-WISE 2015 METRICS =========="
)

display(
    metrics_df[
        (
            metrics_df[
                "scope"
            ]
            == "city"
        )
        & (
            metrics_df[
                "to_epoch"
            ]
            == 2015
        )
    ].reset_index(
        drop=True
    )
)

print()
print(
    "========== CITY-WISE 2020 METRICS =========="
)

display(
    metrics_df[
        (
            metrics_df[
                "scope"
            ]
            == "city"
        )
        & (
            metrics_df[
                "to_epoch"
            ]
            == 2020
        )
    ].reset_index(
        drop=True
    )
)

print(
    "CELL 6 METRICS DISPLAY: SUCCESS"
)

Total metric records: 42

========== OVERALL 2015 AND 2020 METRICS ==========


,to_epoch,scope,city_slug,model,mae,rmse,r2
0,2015,overall,ALL,linear_regression,5904.538118,10700.973279,0.991205
1,2015,overall,ALL,random_forest,6609.759533,15229.473898,0.982187
2,2015,overall,ALL,persistence,7416.663707,16138.461421,0.979997
3,2020,overall,ALL,linear_regression,6961.287531,10977.947370,0.990975
4,2020,overall,ALL,persistence,5503.150680,12107.365945,0.989023
5,2020,overall,ALL,random_forest,9027.187056,18351.458395,0.974781



========== CITY-WISE 2015 METRICS ==========


,to_epoch,scope,city_slug,model,mae,rmse,r2
0,2015,city,bengaluru,linear_regression,7707.098703,14268.247493,0.984710
1,2015,city,bengaluru,random_forest,7423.241535,15289.422135,0.982443
2,2015,city,bengaluru,persistence,12046.245918,22504.077404,0.961964
3,2015,city,chennai_metro,linear_regression,5491.238871,10498.442952,0.989989
4,2015,city,chennai_metro,random_forest,5355.406107,12968.555700,0.984725
5,2015,city,chennai_metro,persistence,7235.572653,16811.241441,0.974331
6,2015,city,delhi_ncr,linear_regression,7741.285416,12032.481587,0.993533
7,2015,city,delhi_ncr,persistence,8839.896735,16933.433546,0.987192
8,2015,city,delhi_ncr,random_forest,12297.261551,24303.068547,0.973617
9,2015,city,hyderabad,linear_regression,5476.279720,9725.103730,0.992362



========== CITY-WISE 2020 METRICS ==========


,to_epoch,scope,city_slug,model,mae,rmse,r2
0,2020,city,bengaluru,linear_regression,7837.404277,11608.833618,0.990015
1,2020,city,bengaluru,persistence,8991.397143,15341.453146,0.982561
2,2020,city,bengaluru,random_forest,11677.823791,20405.696943,0.969148
3,2020,city,chennai_metro,persistence,2413.463673,7309.872862,0.995273
4,2020,city,chennai_metro,linear_regression,6798.714304,10656.731695,0.989954
5,2020,city,chennai_metro,random_forest,8083.520535,17594.175153,0.972617
6,2020,city,delhi_ncr,linear_regression,8626.979540,12719.834073,0.992806
7,2020,city,delhi_ncr,persistence,8962.268571,16770.306312,0.987496
8,2020,city,delhi_ncr,random_forest,14257.687041,25979.124701,0.969992
9,2020,city,hyderabad,linear_regression,6609.732884,9817.819316,0.992410


CELL 6 METRICS DISPLAY: SUCCESS


In [8]:
# ============================================================
# CELL 7: GROWTH-ERROR AND HOTSPOT EVALUATION
# Evaluates actual growth, predicted growth, and top-growth cells.
# ============================================================
payload_b64=base64.b64encode(json.dumps({'lr':LR_PRED,'rf':RF_PRED,'out':EVAL_ROOT+'/growth_hotspot_metrics'}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
P=/tmp/nb13_hotspot.json;S=/tmp/nb13_hotspot.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F,Window
import json,sys
p=json.load(open(sys.argv[1]));s=SparkSession.builder.appName('GHSL-Growth-Hotspot-Evaluation').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 fs=s._jvm.org.apache.hadoop.fs.FileSystem.get(s._jsc.hadoopConfiguration());op=s._jvm.org.apache.hadoop.fs.Path(p['out'])
 if fs.exists(op):raise RuntimeError('Refusing overwrite: '+p['out'])
 rows=[]
 for name,path in [('linear_regression',p['lr']),('random_forest',p['rf'])]:
  d=s.read.parquet(path)
  for epoch in [2015,2020]:
   for city in ['ALL']+sorted(r[0] for r in d.select('city_slug').distinct().collect()):
    x=d.filter(F.col('to_epoch')==epoch);x=x if city=='ALL' else x.filter(F.col('city_slug')==city)
    n=x.count();k=max(1,int(round(n*0.10)))
    actual={r[0] for r in x.orderBy(F.desc('actual_growth_m2')).select('city_slug','grid_cell_id').limit(k).collect()}
    predicted={r[0] for r in x.orderBy(F.desc('predicted_growth_m2')).select(F.concat_ws(':','city_slug','grid_cell_id')).limit(k).collect()}
    # Normalize actual IDs in the same city:key representation.
    actual={r[0] for r in x.orderBy(F.desc('actual_growth_m2')).select(F.concat_ws(':','city_slug','grid_cell_id')).limit(k).collect()}
    inter=len(actual & predicted);precision=inter/max(1,len(predicted));recall=inter/max(1,len(actual));f1=0 if precision+recall==0 else 2*precision*recall/(precision+recall)
    agg=x.agg(F.avg(F.abs('growth_error_m2')).alias('growth_mae'),F.sqrt(F.avg(F.pow('growth_error_m2',2))).alias('growth_rmse'),F.avg('growth_error_m2').alias('growth_bias')).first()
    rows.append({'model':name,'to_epoch':epoch,'city_slug':city,'rows':n,'growth_mae':agg['growth_mae'],'growth_rmse':agg['growth_rmse'],'growth_bias':agg['growth_bias'],'top10_precision':precision,'top10_recall':recall,'top10_f1':f1,'top10_overlap':inter})
 out=s.createDataFrame(rows);out.write.mode('errorifexists').parquet(p['out'])
 print('GROWTH_JSON_BEGIN');print(json.dumps(rows,sort_keys=True));print('GROWTH_JSON_END');print('GROWTH_HOTSPOT_EVALUATION_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 5g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload_b64)
r=run_remote(cmd,7200,show_output=False)
if 'GROWTH_HOTSPOT_EVALUATION_SUCCESS' not in r.stdout:
    print(r.stdout[-12000:]);print(r.stderr[-8000:]);raise RuntimeError('Growth evaluation failed')
GROWTH_METRICS=json.loads(r.stdout.split('GROWTH_JSON_BEGIN',1)[1].split('GROWTH_JSON_END',1)[0].strip())
print('Growth metric records:',len(GROWTH_METRICS))
print('CELL 7: SUCCESS')

Growth metric records: 28
CELL 7: SUCCESS


In [9]:
# ============================================================
# DISPLAY 2015 VS 2020 MODEL COMPARISONS
#
# Uses:
#   - METRICS from Cell 6
#   - GROWTH_METRICS from Cell 7
#
# Does not:
#   - Rerun Spark
#   - Modify HDFS
#   - Retrain models
# ============================================================

import pandas as pd
from IPython.display import display


# ============================================================
# VALIDATE REQUIRED RESULTS
# ============================================================

if "METRICS" not in globals():
    raise RuntimeError(
        "METRICS is unavailable. Run Cell 6 first."
    )

if "GROWTH_METRICS" not in globals():
    raise RuntimeError(
        "GROWTH_METRICS is unavailable. Run Cell 7 first."
    )


metrics_df = pd.DataFrame(
    METRICS
)

growth_df = pd.DataFrame(
    GROWTH_METRICS
)


if len(metrics_df) != 42:
    raise RuntimeError(
        "Expected 42 built-surface metric records, found "
        + str(len(metrics_df))
    )

if len(growth_df) != 28:
    raise RuntimeError(
        "Expected 28 growth metric records, found "
        + str(len(growth_df))
    )


MODEL_LABELS = {
    "persistence": "Persistence baseline",
    "linear_regression": "Linear Regression",
    "random_forest": "Random Forest",
}

CITY_LABELS = {
    "ALL": "All six cities",
    "bengaluru": "Bengaluru",
    "chennai_metro": "Chennai Metropolitan Region",
    "delhi_ncr": "Delhi NCR",
    "hyderabad": "Hyderabad",
    "mumbai_metro": "Mumbai Metropolitan Region",
    "pune_metro": "Pune Metropolitan Region",
}


metrics_df["model_display"] = (
    metrics_df["model"]
    .map(MODEL_LABELS)
    .fillna(metrics_df["model"])
)

metrics_df["city_display"] = (
    metrics_df["city_slug"]
    .map(CITY_LABELS)
    .fillna(metrics_df["city_slug"])
)

growth_df["model_display"] = (
    growth_df["model"]
    .map(MODEL_LABELS)
    .fillna(growth_df["model"])
)

growth_df["city_display"] = (
    growth_df["city_slug"]
    .map(CITY_LABELS)
    .fillna(growth_df["city_slug"])
)


# ============================================================
# 1. OVERALL BUILT-SURFACE METRICS
# ============================================================

overall_metrics = (
    metrics_df[
        metrics_df["scope"]
        == "overall"
    ][
        [
            "to_epoch",
            "model_display",
            "mae",
            "rmse",
            "r2",
        ]
    ]
    .rename(
        columns={
            "to_epoch": "Actual year",
            "model_display": "Model",
            "mae": "MAE (m²)",
            "rmse": "RMSE (m²)",
            "r2": "R²",
        }
    )
    .sort_values(
        [
            "Actual year",
            "RMSE (m²)",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "============================================================"
)
print(
    "OVERALL SIX-CITY COMPARISON"
)
print(
    "============================================================"
)

display(
    overall_metrics.style.format(
        {
            "MAE (m²)": "{:,.2f}",
            "RMSE (m²)": "{:,.2f}",
            "R²": "{:.6f}",
        }
    )
)


# ============================================================
# 2. OVERALL WINNERS BY METRIC
# ============================================================

winner_rows = []

for actual_year in [
    2015,
    2020,
]:
    year_metrics = overall_metrics[
        overall_metrics[
            "Actual year"
        ]
        == actual_year
    ]

    mae_winner = year_metrics.loc[
        year_metrics[
            "MAE (m²)"
        ].idxmin()
    ]

    rmse_winner = year_metrics.loc[
        year_metrics[
            "RMSE (m²)"
        ].idxmin()
    ]

    r2_winner = year_metrics.loc[
        year_metrics[
            "R²"
        ].idxmax()
    ]

    winner_rows.extend(
        [
            {
                "Actual year": actual_year,
                "Metric": "MAE",
                "Best model": mae_winner["Model"],
                "Best value": mae_winner["MAE (m²)"],
            },
            {
                "Actual year": actual_year,
                "Metric": "RMSE",
                "Best model": rmse_winner["Model"],
                "Best value": rmse_winner["RMSE (m²)"],
            },
            {
                "Actual year": actual_year,
                "Metric": "R²",
                "Best model": r2_winner["Model"],
                "Best value": r2_winner["R²"],
            },
        ]
    )


winner_df = pd.DataFrame(
    winner_rows
)


print()
print(
    "============================================================"
)
print(
    "BEST OVERALL MODEL BY METRIC"
)
print(
    "============================================================"
)

display(
    winner_df.style.format(
        {
            "Best value": "{:,.6f}",
        }
    )
)


# ============================================================
# 3. LINEAR REGRESSION: CITY-WISE 2015 VS 2020
# ============================================================

linear_city = (
    metrics_df[
        (
            metrics_df["scope"]
            == "city"
        )
        & (
            metrics_df["model"]
            == "linear_regression"
        )
    ][
        [
            "to_epoch",
            "city_display",
            "mae",
            "rmse",
            "r2",
        ]
    ]
    .rename(
        columns={
            "to_epoch": "Actual year",
            "city_display": "City",
            "mae": "MAE (m²)",
            "rmse": "RMSE (m²)",
            "r2": "R²",
        }
    )
    .sort_values(
        [
            "Actual year",
            "RMSE (m²)",
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "============================================================"
)
print(
    "LINEAR REGRESSION: CITY-WISE PERFORMANCE"
)
print(
    "============================================================"
)

display(
    linear_city.style.format(
        {
            "MAE (m²)": "{:,.2f}",
            "RMSE (m²)": "{:,.2f}",
            "R²": "{:.6f}",
        }
    )
)


# ============================================================
# 4. LINEAR REGRESSION VS PERSISTENCE BY CITY
# ============================================================

city_comparison = (
    metrics_df[
        (
            metrics_df["scope"]
            == "city"
        )
        & (
            metrics_df["model"].isin(
                [
                    "linear_regression",
                    "persistence",
                ]
            )
        )
    ]
    .pivot_table(
        index=[
            "to_epoch",
            "city_display",
        ],
        columns="model",
        values=[
            "mae",
            "rmse",
        ],
    )
    .reset_index()
)


city_comparison.columns = [
    "Actual year",
    "City",
    "Linear MAE",
    "Persistence MAE",
    "Linear RMSE",
    "Persistence RMSE",
]


city_comparison[
    "MAE improvement vs persistence (%)"
] = (
    (
        city_comparison[
            "Persistence MAE"
        ]
        - city_comparison[
            "Linear MAE"
        ]
    )
    / city_comparison[
        "Persistence MAE"
    ]
    * 100.0
)


city_comparison[
    "RMSE improvement vs persistence (%)"
] = (
    (
        city_comparison[
            "Persistence RMSE"
        ]
        - city_comparison[
            "Linear RMSE"
        ]
    )
    / city_comparison[
        "Persistence RMSE"
    ]
    * 100.0
)


city_comparison = city_comparison.sort_values(
    [
        "Actual year",
        "RMSE improvement vs persistence (%)",
    ],
    ascending=[
        True,
        False,
    ],
).reset_index(
    drop=True
)


print()
print(
    "============================================================"
)
print(
    "LINEAR REGRESSION IMPROVEMENT OVER PERSISTENCE"
)
print(
    "Positive improvement means Linear Regression performed better."
)
print(
    "============================================================"
)

display(
    city_comparison.style.format(
        {
            "Linear MAE": "{:,.2f}",
            "Persistence MAE": "{:,.2f}",
            "Linear RMSE": "{:,.2f}",
            "Persistence RMSE": "{:,.2f}",
            "MAE improvement vs persistence (%)": "{:+.2f}%",
            "RMSE improvement vs persistence (%)": "{:+.2f}%",
        }
    )
)


# ============================================================
# 5. OVERALL GROWTH AND HOTSPOT METRICS
# ============================================================

overall_growth = (
    growth_df[
        growth_df["city_slug"]
        == "ALL"
    ][
        [
            "to_epoch",
            "model_display",
            "growth_mae",
            "growth_rmse",
            "growth_bias",
            "top10_precision",
            "top10_recall",
            "top10_f1",
            "top10_overlap",
        ]
    ]
    .rename(
        columns={
            "to_epoch": "Actual year",
            "model_display": "Model",
            "growth_mae": "Growth MAE (m²)",
            "growth_rmse": "Growth RMSE (m²)",
            "growth_bias": "Growth bias (m²)",
            "top10_precision": "Top-10% precision",
            "top10_recall": "Top-10% recall",
            "top10_f1": "Top-10% F1",
            "top10_overlap": "Correct hotspot cells",
        }
    )
    .sort_values(
        [
            "Actual year",
            "Growth RMSE (m²)",
        ]
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "============================================================"
)
print(
    "ACTUAL GROWTH AND TOP-10% HOTSPOT COMPARISON"
)
print(
    "============================================================"
)

display(
    overall_growth.style.format(
        {
            "Growth MAE (m²)": "{:,.2f}",
            "Growth RMSE (m²)": "{:,.2f}",
            "Growth bias (m²)": "{:+,.2f}",
            "Top-10% precision": "{:.2%}",
            "Top-10% recall": "{:.2%}",
            "Top-10% F1": "{:.2%}",
            "Correct hotspot cells": "{:,.0f}",
        }
    )
)


# ============================================================
# 6. CITY-WISE LINEAR GROWTH AND HOTSPOT PERFORMANCE
# ============================================================

linear_growth_city = (
    growth_df[
        (
            growth_df["model"]
            == "linear_regression"
        )
        & (
            growth_df["city_slug"]
            != "ALL"
        )
    ][
        [
            "to_epoch",
            "city_display",
            "growth_mae",
            "growth_rmse",
            "growth_bias",
            "top10_precision",
            "top10_recall",
            "top10_f1",
            "top10_overlap",
        ]
    ]
    .rename(
        columns={
            "to_epoch": "Actual year",
            "city_display": "City",
            "growth_mae": "Growth MAE (m²)",
            "growth_rmse": "Growth RMSE (m²)",
            "growth_bias": "Growth bias (m²)",
            "top10_precision": "Top-10% precision",
            "top10_recall": "Top-10% recall",
            "top10_f1": "Top-10% F1",
            "top10_overlap": "Correct hotspot cells",
        }
    )
    .sort_values(
        [
            "Actual year",
            "Top-10% F1",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "============================================================"
)
print(
    "LINEAR REGRESSION: CITY-WISE GROWTH/HOTSPOT PERFORMANCE"
)
print(
    "============================================================"
)

display(
    linear_growth_city.style.format(
        {
            "Growth MAE (m²)": "{:,.2f}",
            "Growth RMSE (m²)": "{:,.2f}",
            "Growth bias (m²)": "{:+,.2f}",
            "Top-10% precision": "{:.2%}",
            "Top-10% recall": "{:.2%}",
            "Top-10% F1": "{:.2%}",
            "Correct hotspot cells": "{:,.0f}",
        }
    )
)


# ============================================================
# 7. CONCISE TEXT SUMMARY
# ============================================================

print()
print(
    "============================================================"
)
print(
    "AUTOMATIC SUMMARY"
)
print(
    "============================================================"
)


for actual_year in [
    2015,
    2020,
]:
    year_overall = overall_metrics[
        overall_metrics[
            "Actual year"
        ]
        == actual_year
    ]

    best_rmse_row = year_overall.loc[
        year_overall[
            "RMSE (m²)"
        ].idxmin()
    ]

    best_mae_row = year_overall.loc[
        year_overall[
            "MAE (m²)"
        ].idxmin()
    ]

    year_growth = overall_growth[
        overall_growth[
            "Actual year"
        ]
        == actual_year
    ]

    best_hotspot_row = year_growth.loc[
        year_growth[
            "Top-10% F1"
        ].idxmax()
    ]

    print()
    print(
        str(actual_year)
        + ":"
    )

    print(
        " - Best RMSE:",
        best_rmse_row[
            "Model"
        ],
        "with",
        f'{best_rmse_row["RMSE (m²)"]:,.2f} m²',
    )

    print(
        " - Best MAE:",
        best_mae_row[
            "Model"
        ],
        "with",
        f'{best_mae_row["MAE (m²)"]:,.2f} m²',
    )

    print(
        " - Best top-10% hotspot F1:",
        best_hotspot_row[
            "Model"
        ],
        "with",
        f'{best_hotspot_row["Top-10% F1"]:.2%}',
    )


print()
print(
    "DISPLAY COMPARISON CELL: SUCCESS"
)

OVERALL SIX-CITY COMPARISON


,Actual year,Model,MAE (m²),RMSE (m²),R²
0,2015,Linear Regression,"5,904.54","10,700.97",0.991205
1,2015,Random Forest,"6,609.76","15,229.47",0.982187
2,2015,Persistence baseline,"7,416.66","16,138.46",0.979997
3,2020,Linear Regression,"6,961.29","10,977.95",0.990975
4,2020,Persistence baseline,"5,503.15","12,107.37",0.989023
5,2020,Random Forest,"9,027.19","18,351.46",0.974781



BEST OVERALL MODEL BY METRIC


,Actual year,Metric,Best model,Best value
0,2015,MAE,Linear Regression,"5,904.538118"
1,2015,RMSE,Linear Regression,"10,700.973279"
2,2015,R²,Linear Regression,0.991205
3,2020,MAE,Persistence baseline,"5,503.150680"
4,2020,RMSE,Linear Regression,"10,977.947370"
5,2020,R²,Linear Regression,0.990975



LINEAR REGRESSION: CITY-WISE PERFORMANCE


,Actual year,City,MAE (m²),RMSE (m²),R²
0,2015,Mumbai Metropolitan Region,"4,378.18","7,441.62",0.993247
1,2015,Pune Metropolitan Region,"4,633.14","8,861.65",0.988347
2,2015,Hyderabad,"5,476.28","9,725.10",0.992362
3,2015,Chennai Metropolitan Region,"5,491.24","10,498.44",0.989989
4,2015,Delhi NCR,"7,741.29","12,032.48",0.993533
5,2015,Bengaluru,"7,707.10","14,268.25",0.984710
6,2020,Mumbai Metropolitan Region,"5,410.71","9,121.68",0.990027
7,2020,Hyderabad,"6,609.73","9,817.82",0.992410
8,2020,Chennai Metropolitan Region,"6,798.71","10,656.73",0.989954
9,2020,Pune Metropolitan Region,"6,484.18","11,548.69",0.980905


... (3 outputs left out)

,Actual year,Model,Growth MAE (m²),Growth RMSE (m²),Growth bias (m²),Top-10% precision,Top-10% recall,Top-10% F1,Correct hotspot cells
0,2015,Linear Regression,"5,904.54","10,700.97",+854.74,55.54%,55.54%,55.54%,"1,633"
1,2015,Random Forest,"6,609.76","15,229.47",+90.58,55.14%,55.14%,55.14%,"1,621"
2,2020,Linear Regression,"6,961.29","10,977.95","+4,194.39",36.60%,36.60%,36.60%,"1,076"
3,2020,Random Forest,"9,027.19","18,351.46","+2,956.35",28.06%,28.06%,28.06%,825



LINEAR REGRESSION: CITY-WISE GROWTH/HOTSPOT PERFORMANCE


,Actual year,City,Growth MAE (m²),Growth RMSE (m²),Growth bias (m²),Top-10% precision,Top-10% recall,Top-10% F1,Correct hotspot cells
0,2015,Hyderabad,"5,476.28","9,725.10","+1,158.37",61.43%,61.43%,61.43%,301
1,2015,Pune Metropolitan Region,"4,633.14","8,861.65",+148.50,61.43%,61.43%,61.43%,301
2,2015,Chennai Metropolitan Region,"5,491.24","10,498.44",+795.78,60.61%,60.61%,60.61%,297
3,2015,Bengaluru,"7,707.10","14,268.25","-1,285.72",56.73%,56.73%,56.73%,278
4,2015,Mumbai Metropolitan Region,"4,378.18","7,441.62","+2,813.25",54.69%,54.69%,54.69%,268
5,2015,Delhi NCR,"7,741.29","12,032.48","+1,498.23",38.37%,38.37%,38.37%,188
6,2020,Mumbai Metropolitan Region,"5,410.71","9,121.68","+4,264.78",44.90%,44.90%,44.90%,220
7,2020,Hyderabad,"6,609.73","9,817.82","+4,405.52",41.84%,41.84%,41.84%,205
8,2020,Chennai Metropolitan Region,"6,798.71","10,656.73","+6,061.22",41.63%,41.63%,41.63%,204
9,2020,Delhi NCR,"8,626.98","12,719.83","+3,107.37",39.18%,39.18%,39.18%,192



AUTOMATIC SUMMARY

2015:
 - Best RMSE: Linear Regression with 10,700.97 m²
 - Best MAE: Linear Regression with 5,904.54 m²
 - Best top-10% hotspot F1: Linear Regression with 55.54%

2020:
 - Best RMSE: Linear Regression with 10,977.95 m²
 - Best MAE: Persistence baseline with 5,503.15 m²
 - Best top-10% hotspot F1: Linear Regression with 36.60%

DISPLAY COMPARISON CELL: SUCCESS


1. Linear Regression is your strongest overall model

For 2015, Linear Regression wins all three total-built-surface metrics:

MAE: 5,904.54 m²
RMSE: 10,700.97 m²
R2R^2R2: 0.9912

For 2020, Linear Regression still has:

Best RMSE: 10,977.95 m²
Best R2R^2R2: 0.9910

Persistence has the better 2020 MAE:

Persistence: 5,503.15 m²
Linear Regression: 6,961.29 m²

This means Linear Regression controls the larger errors better, while persistence produces smaller absolute errors for many ordinary or slow-changing cells.

2. Random Forest should not be selected

Random Forest is consistently weaker:

Year	Random Forest RMSE	Linear Regression RMSE
2015	15,229	10,701
2020	18,351	10,978

Its hotspot F1 is also lower:

Year	Random Forest	Linear Regression
2015	55.14%	55.54%
2020	28.06%	36.60%

There is no strong reason to carry the current Random Forest into the Jaipur stage.

3. The model generalizes across time, but 2020 is harder

Linear Regression remains numerically strong in 2020, but hotspot performance drops:

2015 hotspot F1: 55.54%
2020 hotspot F1: 36.60%

That is a drop of 18.94 percentage points.

Growth bias also changes substantially:

2015: +854.74 m²
2020: +4,194.39 m²

So by 2020, Linear Regression systematically overpredicts growth.

That is important. The model predicts total built surface accurately overall, but its ability to identify the exact highest-growth cells weakens in the later period.

4. Performance varies materially by city
Strong 2020 results
Mumbai has the lowest Linear Regression RMSE: 9,121.68 m²
Hyderabad has the second-lowest: 9,817.82 m²
Chennai follows at 10,656.73 m²
Highest 2020 errors
Delhi NCR: 12,719.83 m²
Bengaluru: 11,608.83 m²
Pune: 11,548.69 m²

However, low total-built-surface error does not necessarily mean Linear Regression adds value over persistence.

For example, in 2020:

Mumbai persistence RMSE: 3,876.49 m²
Mumbai Linear Regression RMSE: 9,121.68 m²

Persistence is substantially better for Mumbai because its cells appear comparatively stable over that transition.

5. Linear Regression versus persistence is mixed
For 2015

Linear Regression beats persistence by RMSE in five of six cities.

The exception is Mumbai:

Linear Regression RMSE: 7,441.62 m²
Persistence RMSE: 6,202.30 m²
For 2020

Linear Regression beats persistence by RMSE in:

Bengaluru: 24.33% improvement
Delhi NCR: 24.15%
Hyderabad: 12.76%
Pune: 10.85%

Persistence beats Linear Regression in:

Chennai
Mumbai

This suggests a useful final design:

Use Linear Regression as the principal model, but report persistence as a mandatory benchmark and consider a model-selection or blending rule for low-growth cities.

For the project due tomorrow, do not add a complex ensemble now. Document the result and retain Linear Regression as the main learned model.

6. Hotspot results are useful, but moderate rather than exceptional

For 2020, Linear Regression correctly recovers:

1,076 of the actual top 2,940 six-city growth cells
Precision: 36.60%
Recall: 36.60%
F1: 36.60%

The equal precision and recall are expected because both the predicted and observed hotspot sets contain exactly 10% of the evaluated cells.

A random top-10% ranking would have an expected overlap near 10%. Therefore, 36.60% is meaningfully better than random, although it is not strong enough to claim precise hotspot identification.

City-wise 2020 hotspot performance
City	Top-10% F1
Mumbai	44.90%
Hyderabad	41.84%
Chennai	41.63%
Delhi NCR	39.18%
Bengaluru	34.90%
Pune	26.94%

Pune is the weakest city for spatial hotspot localization.

In [10]:
# ============================================================
# CELL 8: SAVE METRIC TABLES TO GOOGLE DRIVE
# ============================================================
import csv
metric_fields=['model','scope','city_slug','to_epoch','mae','rmse','r2']
with (ARTIFACT_ROOT/'built_surface_metrics.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=metric_fields);w.writeheader();w.writerows([{k:x[k] for k in metric_fields} for x in METRICS])
growth_fields=['model','to_epoch','city_slug','rows','growth_mae','growth_rmse','growth_bias','top10_precision','top10_recall','top10_f1','top10_overlap']
with (ARTIFACT_ROOT/'growth_hotspot_metrics.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=growth_fields);w.writeheader();w.writerows([{k:x[k] for k in growth_fields} for x in GROWTH_METRICS])
(ARTIFACT_ROOT/'evaluation_metrics.json').write_text(json.dumps({'built_surface_metrics':METRICS,'growth_hotspot_metrics':GROWTH_METRICS},indent=2,sort_keys=True)+'\n')
print('Saved:',ARTIFACT_ROOT/'built_surface_metrics.csv')
print('Saved:',ARTIFACT_ROOT/'growth_hotspot_metrics.csv')
print('CELL 8: SUCCESS')

Saved: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_evaluation_v1/built_surface_metrics.csv
Saved: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_evaluation_v1/growth_hotspot_metrics.csv
CELL 8: SUCCESS


In [13]:
# ============================================================
# CORRECTED CELL 9:
# EXPORT CELL-LEVEL 2015 AND 2020 PREDICTIONS
#
# Fix:
#   - Excludes binary geometry_wkb from CSV output
#   - Retains grid bounds for later mapping
#
# Safety:
#   - Reads existing HDFS prediction datasets
#   - Writes only temporary CSV files under /tmp on EC2
#   - Copies compact CSV outputs to Google Drive
#   - Does not modify HDFS
#   - Safely replaces only local Drive CSV exports
# ============================================================

required_names = [
    "REMOTE_EXEC",
    "ARTIFACT_ROOT",
    "run_remote",
]

for required_name in required_names:
    if required_name not in globals():
        raise RuntimeError(
            required_name
            + " is unavailable. Run the earlier cells first."
        )


# ============================================================
# REMOTE EXPORT JOB
# ============================================================

export_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

EXPORT_ROOT="/tmp/nb13_eval_exports"
SCRIPT="/tmp/nb13_corrected_exports.py"

echo "============================================================"
echo "CORRECTED SIX-CITY PREDICTION EXPORT"
echo "============================================================"

rm -rf "$EXPORT_ROOT"
mkdir -p "$EXPORT_ROOT"

cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import (
    SparkSession,
    functions as F,
)


spark = (
    SparkSession.builder
    .appName(
        "GHSL-Corrected-Evaluation-Exports"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel(
    "WARN"
)


model_inputs = [
    (
        "linear_regression",
        (
            "/data/ghsl/modeling/evaluation/"
            "six_city_2015_2020_v1/"
            "linear_regression_predictions"
        ),
    ),
    (
        "random_forest",
        (
            "/data/ghsl/modeling/evaluation/"
            "six_city_2015_2020_v1/"
            "random_forest_predictions"
        ),
    ),
]


export_columns = [
    "city_slug",
    "grid_cell_id",
    "grid_1km_row",
    "grid_1km_col",
    "grid_min_x",
    "grid_max_x",
    "grid_min_y",
    "grid_max_y",
    "geometry_crs",
    "from_epoch",
    "to_epoch",
    "current_built_surface_m2",
    "label",
    "model_name",
    "model_prediction",
    "persistence_prediction",
    "model_error",
    "model_abs_error",
    "persistence_error",
    "persistence_abs_error",
    "actual_growth_m2",
    "predicted_growth_m2",
    "growth_error_m2",
]


try:
    for (
        model_name,
        prediction_path,
    ) in model_inputs:
        dataframe = (
            spark.read
            .parquet(
                prediction_path
            )
        )

        missing_columns = [
            column_name
            for column_name
            in export_columns
            if column_name
            not in dataframe.columns
        ]

        if missing_columns:
            raise RuntimeError(
                model_name
                + " prediction dataset is missing "
                + "required export columns: "
                + repr(
                    missing_columns
                )
            )

        # geometry_wkb is intentionally excluded because it is binary.
        # Grid bounds are retained and are sufficient to reconstruct
        # each 1 km cell for later mapping.
        export_frame = (
            dataframe
            .select(
                *export_columns
            )
        )

        for actual_year in [
            2015,
            2020,
        ]:
            year_frame = (
                export_frame
                .filter(
                    F.col(
                        "to_epoch"
                    )
                    == actual_year
                )
            )

            row_count = (
                year_frame.count()
            )

            if row_count != 29400:
                raise RuntimeError(
                    model_name
                    + " "
                    + str(actual_year)
                    + ": expected 29,400 rows, found "
                    + str(row_count)
                )

            destination = (
                "/tmp/nb13_eval_exports/"
                + model_name
                + "_"
                + str(actual_year)
            )

            (
                year_frame
                .orderBy(
                    "city_slug",
                    "grid_cell_id",
                )
                .coalesce(1)
                .write
                .mode(
                    "overwrite"
                )
                .option(
                    "header",
                    True,
                )
                .option(
                    "nullValue",
                    "",
                )
                .csv(
                    destination
                )
            )

            print(
                "EXPORTED:",
                model_name,
                actual_year,
                "rows=",
                row_count,
                "destination=",
                destination,
            )


    print(
        "CORRECTED_CELL_9_SPARK_EXPORT_SUCCESS"
    )


finally:
    spark.stop()
PY_REMOTE


spark-submit \
    --master local[1] \
    --driver-memory 4g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    "$SCRIPT"


echo
echo "========== EXPORTED FILE INVENTORY =========="

find "$EXPORT_ROOT" \
    -maxdepth 3 \
    -type f \
    -printf '%p %s bytes\n' \
    | sort


for export_name in \
    linear_regression_2015 \
    linear_regression_2020 \
    random_forest_2015 \
    random_forest_2020
do
    csv_count="$(
        find \
            "$EXPORT_ROOT/$export_name" \
            -type f \
            -name 'part-*.csv' \
        | wc -l
    )"

    if [ "$csv_count" -ne 1 ]; then
        echo "ERROR: Expected exactly one CSV part file for:"
        echo "$export_name"
        echo "Found: $csv_count"
        exit 30
    fi

    csv_path="$(
        find \
            "$EXPORT_ROOT/$export_name" \
            -type f \
            -name 'part-*.csv' \
        | head -1
    )"

    if [ ! -s "$csv_path" ]; then
        echo "ERROR: Exported CSV is empty:"
        echo "$csv_path"
        exit 31
    fi

    line_count="$(
        wc -l \
            < "$csv_path"
    )"

    if [ "$line_count" -ne 29401 ]; then
        echo "ERROR: Expected 29,401 CSV lines for:"
        echo "$export_name"
        echo "Found: $line_count"
        exit 32
    fi

    echo "VALIDATED: $export_name lines=$line_count"
done


rm -f "$SCRIPT"

echo
echo "Temporary export root retained for Drive transfer: YES"
echo "HDFS modified: NO"
echo "CORRECTED_CELL_9_REMOTE_EXPORT_SUCCESS"
'''


export_result = run_remote(
    export_command,
    timeout=3600,
    show_output=True,
)


if (
    "CORRECTED_CELL_9_SPARK_EXPORT_SUCCESS"
    not in export_result.stdout
):
    raise RuntimeError(
        "The corrected Spark CSV export "
        "did not produce its success marker."
    )


if (
    "CORRECTED_CELL_9_REMOTE_EXPORT_SUCCESS"
    not in export_result.stdout
):
    raise RuntimeError(
        "The corrected remote export validation "
        "did not produce its success marker."
    )


# ============================================================
# TRANSFER THE FOUR CSV FILES TO GOOGLE DRIVE
# ============================================================

export_specs = [
    (
        "linear_regression",
        2015,
    ),
    (
        "linear_regression",
        2020,
    ),
    (
        "random_forest",
        2015,
    ),
    (
        "random_forest",
        2020,
    ),
]


exported_drive_files = []


for (
    model_name,
    actual_year,
) in export_specs:
    remote_directory = (
        "/tmp/nb13_eval_exports/"
        + model_name
        + "_"
        + str(actual_year)
    )

    read_command = (
        "set -e; "
        "CSV_PATH=\"$(find "
        + "'"
        + remote_directory
        + "' "
        + "-type f -name 'part-*.csv' "
        + "| head -1)\"; "
        + "test -n \"$CSV_PATH\"; "
        + "test -s \"$CSV_PATH\"; "
        + "cat \"$CSV_PATH\""
    )

    csv_result = run_remote(
        read_command,
        timeout=1200,
        show_output=False,
    )

    if not csv_result.stdout:
        raise RuntimeError(
            "No CSV content was returned for "
            + model_name
            + " "
            + str(actual_year)
        )

    local_path = (
        ARTIFACT_ROOT
        / (
            model_name
            + "_"
            + str(actual_year)
            + "_cell_predictions.csv"
        )
    )

    local_path.write_text(
        csv_result.stdout,
        encoding="utf-8",
    )

    if not local_path.is_file():
        raise RuntimeError(
            "Drive export was not created: "
            + str(local_path)
        )

    if local_path.stat().st_size <= 0:
        raise RuntimeError(
            "Drive export is empty: "
            + str(local_path)
        )

    # Header + 29,400 data rows.
    local_line_count = sum(
        1
        for _ in local_path.open(
            "r",
            encoding="utf-8",
        )
    )

    if local_line_count != 29401:
        raise RuntimeError(
            "Drive export has an incorrect line count. "
            + str(local_path)
            + ": expected 29,401, found "
            + str(local_line_count)
        )

    exported_drive_files.append(
        {
            "model": model_name,
            "actual_year": actual_year,
            "path": str(local_path),
            "size_bytes": (
                local_path.stat().st_size
            ),
            "line_count": (
                local_line_count
            ),
        }
    )

    print(
        "Saved:",
        local_path,
    )

    print(
        "Rows:",
        local_line_count - 1,
    )

    print(
        "Size:",
        f"{local_path.stat().st_size / (1024 * 1024):.2f} MB",
    )


# ============================================================
# SAVE EXPORT MANIFEST
# ============================================================

export_manifest_path = (
    ARTIFACT_ROOT
    / "cell_prediction_exports.json"
)


export_manifest = {
    "created_at_utc": (
        time.strftime(
            "%Y-%m-%dT%H:%M:%SZ",
            time.gmtime(),
        )
    ),
    "models": [
        "linear_regression",
        "random_forest",
    ],
    "actual_years": [
        2015,
        2020,
    ],
    "rows_per_export": 29400,
    "geometry_wkb_exported": False,
    "geometry_wkb_exclusion_reason": (
        "Spark CSV does not support binary columns."
    ),
    "mapping_fields_retained": [
        "grid_1km_row",
        "grid_1km_col",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "geometry_crs",
    ],
    "files": exported_drive_files,
    "hdfs_modified": False,
}


export_manifest_path.write_text(
    json.dumps(
        export_manifest,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Export manifest:",
    export_manifest_path,
)

print(
    "CSV exports created:",
    len(
        exported_drive_files
    ),
)

print(
    "Rows per CSV:",
    29400,
)

print(
    "geometry_wkb exported: NO"
)

print(
    "Grid bounds retained for mapping: YES"
)

print(
    "HDFS modified: NO"
)

print(
    "CELL 9: SUCCESS"
)

CORRECTED SIX-CITY PREDICTION EXPORT
EXPORTED: linear_regression 2015 rows= 29400 destination= /tmp/nb13_eval_exports/linear_regression_2015
EXPORTED: linear_regression 2020 rows= 29400 destination= /tmp/nb13_eval_exports/linear_regression_2020
EXPORTED: random_forest 2015 rows= 29400 destination= /tmp/nb13_eval_exports/random_forest_2015
EXPORTED: random_forest 2020 rows= 29400 destination= /tmp/nb13_eval_exports/random_forest_2020
CORRECTED_CELL_9_SPARK_EXPORT_SUCCESS

========== EXPORTED FILE INVENTORY ==========
ERROR: Expected exactly one CSV part file for:
linear_regression_2015
Found: 0

REMOTE STDERR:
26/11/15 06:06:58 INFO SparkContext: Running Spark version 3.5.9
26/11/15 06:06:58 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/15 06:06:58 INFO SparkContext: Java version 11.0.32.1
26/11/15 06:06:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/15 06:06:59 INFO ResourceUtils: ===

RuntimeError: Remote command failed with return code 30

In [12]:
# ============================================================
# CELL 9 DIAGNOSTIC: SHOW EXPORT FAILURE AND PARTIAL OUTPUTS
#
# Read-only:
#   - Does not rerun Spark
#   - Does not delete export directories
#   - Does not modify HDFS
# ============================================================

diagnostic_command = r'''
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "============================================================"
echo "CELL 9 EXPORT FAILURE DIAGNOSTIC"
echo "============================================================"

echo
echo "========== TEMPORARY EXPORT SCRIPT =========="

if [ -f /tmp/nb13_exports.py ]; then
    echo "FOUND: /tmp/nb13_exports.py"
    ls -lh /tmp/nb13_exports.py

    echo
    echo "Script contents:"
    sed -n '1,240p' /tmp/nb13_exports.py
else
    echo "MISSING: /tmp/nb13_exports.py"
fi

echo
echo "========== TEMPORARY EXPORT ROOT =========="

if [ -d /tmp/nb13_eval_exports ]; then
    echo "FOUND: /tmp/nb13_eval_exports"

    find /tmp/nb13_eval_exports \
        -maxdepth 3 \
        -type f \
        -printf '%p %s bytes\n' \
        | sort
else
    echo "MISSING: /tmp/nb13_eval_exports"
fi

echo
echo "========== LINEAR REGRESSION HDFS PREDICTIONS =========="

if hdfs dfs -test -e \
    /data/ghsl/modeling/evaluation/six_city_2015_2020_v1/linear_regression_predictions
then
    echo "LINEAR_REGRESSION_PREDICTIONS: PRESENT"

    hdfs dfs -count -h \
        /data/ghsl/modeling/evaluation/six_city_2015_2020_v1/linear_regression_predictions
else
    echo "LINEAR_REGRESSION_PREDICTIONS: MISSING"
fi

echo
echo "========== RANDOM FOREST HDFS PREDICTIONS =========="

if hdfs dfs -test -e \
    /data/ghsl/modeling/evaluation/six_city_2015_2020_v1/random_forest_predictions
then
    echo "RANDOM_FOREST_PREDICTIONS: PRESENT"

    hdfs dfs -count -h \
        /data/ghsl/modeling/evaluation/six_city_2015_2020_v1/random_forest_predictions
else
    echo "RANDOM_FOREST_PREDICTIONS: MISSING"
fi

echo
echo "========== DISK STATUS =========="

df -h / /tmp

echo
echo "CELL_9_DIAGNOSTIC_COMPLETE"
'''

diagnostic_result = run_remote(
    diagnostic_command,
    timeout=300,
    show_output=True,
)

print(
    "Return code:",
    diagnostic_result.returncode,
)

print(
    "CELL 9 DIAGNOSTIC: SUCCESS"
)

CELL 9 EXPORT FAILURE DIAGNOSTIC

========== TEMPORARY EXPORT SCRIPT ==========
FOUND: /tmp/nb13_exports.py
-rw-rw-r-- 1 ubuntu ubuntu 583 Nov 15 06:05 /tmp/nb13_exports.py

Script contents:
from pyspark.sql import SparkSession
s=SparkSession.builder.appName('GHSL-Evaluation-Exports').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 for name,path in [('linear_regression','/data/ghsl/modeling/evaluation/six_city_2015_2020_v1/linear_regression_predictions'),('random_forest','/data/ghsl/modeling/evaluation/six_city_2015_2020_v1/random_forest_predictions')]:
  d=s.read.parquet(path)
  for epoch in [2015,2020]: d.filter(d.to_epoch==epoch).coalesce(1).write.mode('overwrite').option('header',True).csv('/tmp/nb13_eval_exports/'+name+'_'+str(epoch))
finally:s.stop()

========== TEMPORARY EXPORT ROOT ==========
FOUND: /tmp/nb13_eval_exports

========== LINEAR REGRESSION HDFS PREDICTIONS ==========
LINEAR_REGRESSION_PREDICTIONS: PRESENT
          15           13              5.6 M /data/ghs

In [14]:
# ============================================================
# CELL 9 RECOVERY:
# TRANSFER EXISTING HDFS CSV EXPORTS TO GOOGLE DRIVE
#
# Why this is needed:
#   Spark interpreted /tmp/nb13_eval_exports as an HDFS path,
#   not an EC2 local-filesystem path.
#
# This cell:
#   - Does not rerun Spark
#   - Does not regenerate predictions
#   - Does not modify source prediction datasets
#   - Reads the already-created temporary HDFS CSV files
#   - Saves and validates four CSV files in Google Drive
#   - Removes only the temporary HDFS export root after success
# ============================================================

required_names = [
    "ARTIFACT_ROOT",
    "run_remote",
]

for required_name in required_names:
    if required_name not in globals():
        raise RuntimeError(
            required_name
            + " is unavailable. Run the earlier cells first."
        )


HDFS_TEMP_EXPORT_ROOT = (
    "/tmp/nb13_eval_exports"
)


export_specs = [
    {
        "model": "linear_regression",
        "actual_year": 2015,
        "hdfs_directory": (
            HDFS_TEMP_EXPORT_ROOT
            + "/linear_regression_2015"
        ),
        "drive_filename": (
            "linear_regression_2015_cell_predictions.csv"
        ),
    },
    {
        "model": "linear_regression",
        "actual_year": 2020,
        "hdfs_directory": (
            HDFS_TEMP_EXPORT_ROOT
            + "/linear_regression_2020"
        ),
        "drive_filename": (
            "linear_regression_2020_cell_predictions.csv"
        ),
    },
    {
        "model": "random_forest",
        "actual_year": 2015,
        "hdfs_directory": (
            HDFS_TEMP_EXPORT_ROOT
            + "/random_forest_2015"
        ),
        "drive_filename": (
            "random_forest_2015_cell_predictions.csv"
        ),
    },
    {
        "model": "random_forest",
        "actual_year": 2020,
        "hdfs_directory": (
            HDFS_TEMP_EXPORT_ROOT
            + "/random_forest_2020"
        ),
        "drive_filename": (
            "random_forest_2020_cell_predictions.csv"
        ),
    },
]


# ============================================================
# VERIFY THE FOUR HDFS EXPORT DIRECTORIES
# ============================================================

verification_lines = [
    "set -e",
    "source /home/ubuntu/ghsl-sedona-env.sh",
    "export HADOOP_HOME=/usr/local/hadoop",
    'export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"',
    "",
    'echo "============================================================"',
    'echo "VERIFYING TEMPORARY HDFS CSV EXPORTS"',
    'echo "============================================================"',
    "",
]


for export_spec in export_specs:
    hdfs_directory = (
        export_spec[
            "hdfs_directory"
        ]
    )

    verification_lines.extend(
        [
            'echo',
            (
                'echo "Directory: '
                + hdfs_directory
                + '"'
            ),
            (
                "hdfs dfs -test -d '"
                + hdfs_directory
                + "'"
            ),
            (
                "hdfs dfs -ls '"
                + hdfs_directory
                + "'"
            ),
            (
                "CSV_COUNT=\"$(hdfs dfs -find '"
                + hdfs_directory
                + "' -name 'part-*.csv' "
                + "| wc -l)\""
            ),
            (
                'if [ "$CSV_COUNT" -ne 1 ]; then '
                + 'echo "Expected exactly one CSV part file, '
                + 'found $CSV_COUNT"; exit 31; fi'
            ),
            (
                "CSV_PATH=\"$(hdfs dfs -find '"
                + hdfs_directory
                + "' -name 'part-*.csv' "
                + "| head -1)\""
            ),
            (
                'echo "CSV path: $CSV_PATH"'
            ),
            (
                'LINE_COUNT="$(hdfs dfs -cat '
                + '"$CSV_PATH" | wc -l)"'
            ),
            (
                'echo "Line count: $LINE_COUNT"'
            ),
            (
                'if [ "$LINE_COUNT" -ne 29401 ]; then '
                + 'echo "Expected 29,401 lines, found '
                + '$LINE_COUNT"; exit 32; fi'
            ),
        ]
    )


verification_lines.extend(
    [
        "",
        'echo "HDFS_TEMP_EXPORT_VERIFICATION_SUCCESS"',
    ]
)


verification_command = "\n".join(
    verification_lines
)


verification_result = run_remote(
    verification_command,
    timeout=1200,
    show_output=True,
)


if (
    "HDFS_TEMP_EXPORT_VERIFICATION_SUCCESS"
    not in verification_result.stdout
):
    raise RuntimeError(
        "The temporary HDFS CSV exports "
        "did not pass verification."
    )


# ============================================================
# TRANSFER EACH HDFS CSV TO GOOGLE DRIVE
# ============================================================

exported_drive_files = []


for export_spec in export_specs:
    model_name = (
        export_spec[
            "model"
        ]
    )

    actual_year = (
        export_spec[
            "actual_year"
        ]
    )

    hdfs_directory = (
        export_spec[
            "hdfs_directory"
        ]
    )

    drive_path = (
        ARTIFACT_ROOT
        / export_spec[
            "drive_filename"
        ]
    )

    read_command = (
        "set -e\n"
        "source /home/ubuntu/ghsl-sedona-env.sh\n"
        "export HADOOP_HOME=/usr/local/hadoop\n"
        'export PATH="${HADOOP_HOME}/bin:'
        '${HADOOP_HOME}/sbin:${PATH}"\n'
        "CSV_PATH=\"$(hdfs dfs -find '"
        + hdfs_directory
        + "' -name 'part-*.csv' | head -1)\"\n"
        'test -n "$CSV_PATH"\n'
        'hdfs dfs -test -e "$CSV_PATH"\n'
        'hdfs dfs -cat "$CSV_PATH"\n'
    )

    csv_result = run_remote(
        read_command,
        timeout=1800,
        show_output=False,
    )

    csv_text = (
        csv_result.stdout
    )

    if not csv_text:
        raise RuntimeError(
            "No CSV content was returned for "
            + model_name
            + " "
            + str(actual_year)
        )

    drive_path.write_text(
        csv_text,
        encoding="utf-8",
    )

    if not drive_path.is_file():
        raise RuntimeError(
            "Drive CSV was not created: "
            + str(drive_path)
        )

    if drive_path.stat().st_size <= 0:
        raise RuntimeError(
            "Drive CSV is empty: "
            + str(drive_path)
        )

    with drive_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        line_count = sum(
            1
            for _ in handle
        )

    if line_count != 29401:
        raise RuntimeError(
            "Incorrect Drive CSV line count for "
            + str(drive_path)
            + ". Expected 29,401, found "
            + str(line_count)
        )

    header = (
        drive_path
        .open(
            "r",
            encoding="utf-8",
        )
        .readline()
        .strip()
        .split(",")
    )

    required_export_columns = {
        "city_slug",
        "grid_cell_id",
        "from_epoch",
        "to_epoch",
        "label",
        "model_prediction",
        "persistence_prediction",
        "model_error",
        "model_abs_error",
        "actual_growth_m2",
        "predicted_growth_m2",
        "growth_error_m2",
    }

    missing_header_columns = sorted(
        required_export_columns
        - set(header)
    )

    if missing_header_columns:
        raise RuntimeError(
            "Drive CSV is missing expected columns: "
            + repr(
                missing_header_columns
            )
        )

    if "geometry_wkb" in header:
        raise RuntimeError(
            "Binary geometry_wkb unexpectedly appears "
            "in the CSV export."
        )

    file_record = {
        "model": model_name,
        "actual_year": actual_year,
        "path": str(
            drive_path
        ),
        "size_bytes": (
            drive_path.stat().st_size
        ),
        "line_count": (
            line_count
        ),
        "data_rows": (
            line_count - 1
        ),
    }

    exported_drive_files.append(
        file_record
    )

    print(
        "Saved:",
        drive_path,
    )

    print(
        "Data rows:",
        line_count - 1,
    )

    print(
        "Size:",
        (
            f"{drive_path.stat().st_size / (1024 * 1024):.2f} MB"
        ),
    )


# ============================================================
# SAVE THE DRIVE EXPORT MANIFEST
# ============================================================

export_manifest = {
    "created_at_utc": (
        time.strftime(
            "%Y-%m-%dT%H:%M:%SZ",
            time.gmtime(),
        )
    ),
    "source": (
        "temporary_hdfs_csv_exports"
    ),
    "temporary_hdfs_root": (
        HDFS_TEMP_EXPORT_ROOT
    ),
    "models": [
        "linear_regression",
        "random_forest",
    ],
    "actual_years": [
        2015,
        2020,
    ],
    "rows_per_export": 29400,
    "geometry_wkb_exported": False,
    "geometry_wkb_exclusion_reason": (
        "Spark CSV does not support binary columns."
    ),
    "mapping_fields_retained": [
        "grid_1km_row",
        "grid_1km_col",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "geometry_crs",
    ],
    "files": (
        exported_drive_files
    ),
    "source_prediction_datasets_modified": False,
}


export_manifest_path = (
    ARTIFACT_ROOT
    / "cell_prediction_exports.json"
)


export_manifest_path.write_text(
    json.dumps(
        export_manifest,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


if not export_manifest_path.is_file():
    raise RuntimeError(
        "The export manifest was not created."
    )


# ============================================================
# REMOVE ONLY THE TEMPORARY HDFS CSV EXPORT DIRECTORY
# ============================================================

cleanup_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

TEMP_EXPORT_ROOT="/tmp/nb13_eval_exports"

hdfs dfs -test -d "$TEMP_EXPORT_ROOT"

hdfs dfs -rm \
    -r \
    -skipTrash \
    "$TEMP_EXPORT_ROOT"

if hdfs dfs -test -e "$TEMP_EXPORT_ROOT"; then
    echo "Temporary HDFS export root still exists."
    exit 40
fi

echo "TEMPORARY_HDFS_EXPORT_CLEANUP_SUCCESS"
'''


cleanup_result = run_remote(
    cleanup_command,
    timeout=600,
    show_output=True,
)


if (
    "TEMPORARY_HDFS_EXPORT_CLEANUP_SUCCESS"
    not in cleanup_result.stdout
):
    raise RuntimeError(
        "Temporary HDFS export cleanup failed."
    )


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "CSV exports created:",
    len(
        exported_drive_files
    ),
)

print(
    "Rows per export:",
    29400,
)

print(
    "All Drive CSV line counts validated: YES"
)

print(
    "Required CSV columns validated: YES"
)

print(
    "geometry_wkb exported: NO"
)

print(
    "Grid bounds retained for mapping: YES"
)

print(
    "Temporary HDFS CSV directory removed: YES"
)

print(
    "Source prediction datasets modified: NO"
)

print(
    "Export manifest:",
    export_manifest_path,
)

print(
    "CELL 9: SUCCESS"
)

VERIFYING TEMPORARY HDFS CSV EXPORTS

Directory: /tmp/nb13_eval_exports/linear_regression_2015
Found 2 items
-rw-r--r--   1 ubuntu supergroup          0 2026-11-15 06:07 /tmp/nb13_eval_exports/linear_regression_2015/_SUCCESS
-rw-r--r--   1 ubuntu supergroup    7416258 2026-11-15 06:07 /tmp/nb13_eval_exports/linear_regression_2015/part-00000-0b852497-0f6b-427a-b42d-05a449d320a0-c000.csv
CSV path: /tmp/nb13_eval_exports/linear_regression_2015/part-00000-0b852497-0f6b-427a-b42d-05a449d320a0-c000.csv
Line count: 29401

Directory: /tmp/nb13_eval_exports/linear_regression_2020
Found 2 items
-rw-r--r--   1 ubuntu supergroup          0 2026-11-15 06:07 /tmp/nb13_eval_exports/linear_regression_2020/_SUCCESS
-rw-r--r--   1 ubuntu supergroup    7424144 2026-11-15 06:07 /tmp/nb13_eval_exports/linear_regression_2020/part-00000-8afaa0c2-0493-4a2c-a8c5-12685fea47be-c000.csv
CSV path: /tmp/nb13_eval_exports/linear_regression_2020/part-00000-8afaa0c2-0493-4a2c-a8c5-12685fea47be-c000.csv
Line count: 294

All four cell-level prediction files are now safely stored in Google Drive:

Plain Text
/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_evaluation_v1/
Exported files
linear_regression_2015_cell_predictions.csv: 29,400 rows
linear_regression_2020_cell_predictions.csv: 29,400 rows
random_forest_2015_cell_predictions.csv: 29,400 rows
random_forest_2020_cell_predictions.csv: 29,400 rows

Each file retains:

City and grid-cell identity
Actual and predicted built surface
Persistence prediction
Absolute and signed errors
Actual and predicted growth
Growth error
Grid row and column
Coordinate bounds for mapping
CRS information

In [15]:
# ============================================================
# CELL 10: GENERATE COMPARISON FIGURES IN COLAB
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
FIG=ARTIFACT_ROOT/'figures';FIG.mkdir(exist_ok=True)
metrics_df=pd.DataFrame(METRICS)
overall=metrics_df[metrics_df.scope=='overall'].copy()
for metric in ['mae','rmse','r2']:
    pivot=overall.pivot(index='model',columns='to_epoch',values=metric)
    ax=pivot.plot(kind='bar',figsize=(9,5),title=f'Overall {metric.upper()}: actual versus model evaluation')
    ax.set_xlabel('Model');ax.set_ylabel(metric.upper());plt.xticks(rotation=0);plt.tight_layout();plt.savefig(FIG/f'overall_{metric}.png',dpi=180);plt.close()
city=metrics_df[(metrics_df.scope=='city') & (metrics_df.model=='linear_regression')]
for metric in ['mae','rmse','r2']:
    pivot=city.pivot(index='city_slug',columns='to_epoch',values=metric)
    ax=pivot.plot(kind='bar',figsize=(11,5),title=f'Linear Regression city-wise {metric.upper()}')
    ax.set_xlabel('City');ax.set_ylabel(metric.upper());plt.xticks(rotation=25,ha='right');plt.tight_layout();plt.savefig(FIG/f'linear_city_{metric}.png',dpi=180);plt.close()
print('Figures:',len(list(FIG.glob('*.png'))))
print('CELL 10: SUCCESS')

Figures: 6
CELL 10: SUCCESS


In [16]:
# ============================================================
# CELL 11: FINAL EVALUATION MANIFEST AND COMPLETION GATE
# ============================================================
def overall(model,epoch):
    return next(x for x in METRICS if x['model']==model and x['scope']=='overall' and x['to_epoch']==epoch)
manifest={
 'created_at_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
 'purpose':'Compare predicted versus actual 2015 and 2020 built surface for the six existing cities',
 'training_through':2010,
 'validation_period':'2010->2015',
 'test_period':'2015->2020',
 'cities':EXPECTED_CITIES,
 'jaipur_included':False,
 'models':['persistence','linear_regression','random_forest'],
 'target':'next built_surface_total_m2',
 'validation_rows':29400,
 'test_rows':29400,
 'overall_2015':{m:overall(m,2015) for m in ['persistence','linear_regression','random_forest']},
 'overall_2020':{m:overall(m,2020) for m in ['persistence','linear_regression','random_forest']},
 'hdfs_outputs':{'linear_predictions':LR_PRED,'random_forest_predictions':RF_PRED,'metric_tables':SUMMARY_OUT},
 'source_datasets_modified':False,
}
path=ARTIFACT_ROOT/'six_city_evaluation_manifest.json';path.write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
check=run_remote("source /home/ubuntu/ghsl-sedona-env.sh; export HADOOP_HOME=/usr/local/hadoop; export PATH=$HADOOP_HOME/bin:$PATH; for p in '"+LR_PRED+"' '"+RF_PRED+"' '"+SUMMARY_OUT+"'; do hdfs dfs -test -e \"$p\" || exit 1; done; echo EVALUATION_HDFS_OUTPUTS_PASS",300)
print('2015 actual-vs-predicted comparison: COMPLETE')
print('2020 actual-vs-predicted comparison: COMPLETE')
print('Overall metrics: COMPLETE')
print('City-wise metrics: COMPLETE')
print('Growth-error metrics: COMPLETE')
print('Top-10-percent hotspot evaluation: COMPLETE')
print('Cell-level prediction exports: COMPLETE')
print('Jaipur included: NO')
print('Source HDFS datasets modified: NO')
print('NOTEBOOK_13_SIX_CITY_EVALUATION_SUCCESS')
print('CELL 11: SUCCESS')

EVALUATION_HDFS_OUTPUTS_PASS

REMOTE STDERR:

2015 actual-vs-predicted comparison: COMPLETE
2020 actual-vs-predicted comparison: COMPLETE
Overall metrics: COMPLETE
City-wise metrics: COMPLETE
Growth-error metrics: COMPLETE
Top-10-percent hotspot evaluation: COMPLETE
Cell-level prediction exports: COMPLETE
Jaipur included: NO
Source HDFS datasets modified: NO
NOTEBOOK_13_SIX_CITY_EVALUATION_SUCCESS
CELL 11: SUCCESS


## Next step

After reviewing the 2015 and 2020 results, choose the final model. Only then create Notebook 14 to retrain that model through 2020 and evaluate Jaipur as the unseen city.